# एसआरए लर्निंग और रूटिंग डेमो (कॉपी टास्क)

इस नोटबुक में, हम एक एसआरए मॉडल को एक साधारण कार्य (कॉपी कार्य: इनपुट अनुक्रम को यथावत आउटपुट) पर प्रशिक्षित करते हैं।
हम कल्पना करते हैं कि **प्रशिक्षण से पहले और बाद में रूटिंग (प्रत्येक इनपुट टोकन किस सिनैप्स को सौंपा गया है) कैसे बदलती है**, और देखते हैं कि एसआरए कार्य-विशेष तरीके से विशेषज्ञों (सिनैप्स) का उपयोग करना कैसे सीखता है।

## 1. पर्यावरण सेटअप
यदि आप Google Colab पर हैं, तो नीचे दिया गया सेल चलाएँ।

In [ ]:
import sys
if 'google.colab' in sys.modules:
    !git clone https://github.com/JunSuzukiJapan/SynapticRouter.git
    %cd SynapticRouter
    !pip install torch matplotlib seaborn

sys.path.append('.')
sys.path.append('./src')
if 'google.colab' not in sys.modules:
    sys.path.append('..')
    sys.path.append('../src')


## 2. पुस्तकालय आयात करें और मॉडल तैयार करें

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import seaborn as sns
import torch.nn.functional as F

from src.sra_gpu_models import MoESRAModel
class MoESRAConfig:
    def __init__(self, **kwargs):
        for k, v in kwargs.items():
            setattr(self, k, v)
from src.sra_experiment import make_batch, make_optimizer, load_balance_loss

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Use a slightly smaller model so that training converges quickly
config = MoESRAConfig(
    vocab_size=20,
    d_model=64,
    n_layers=2,
    n_heads=2,
    num_synapses=4,  # 4 experts (Synapses)
    k=2,
    max_seq_len=32
)
model = MoESRAModel(config.vocab_size, config.d_model, config.n_layers, config.num_synapses, config.k, syn_hidden=128).to(device)
optimizer = make_optimizer(model, lr=0.005)

## 3. विज़ुअलाइज़ेशन फ़ंक्शन को परिभाषित करें
रूटिंग वेट को हीटमैप के रूप में प्रदर्शित करने के लिए एक फ़ंक्शन तैयार करें।

In [ ]:
def visualize_routing(model, task="copy", title_prefix=""):
    model.eval()
    # Build an evaluation batch (sequence length 10)
    x, y = make_batch(task, batch_size=1, min_len=10, max_len=10, device=device)
    
    with torch.no_grad():
        y_in = torch.cat([torch.full((y.size(0), 1), 1, dtype=torch.long, device=device), y[:, :-1]], dim=1)
        outputs, routing_weights, _ = model(x, y_in)
        
    # Get the routing weights from the first layer
    layer_idx = 0
    weights = routing_weights[layer_idx][0].cpu().numpy()
    
    plt.figure(figsize=(8, 5))
    sns.heatmap(weights, cmap="YlGnBu", annot=True, fmt=".2f")
    plt.title(f"{title_prefix} Routing Weights (Layer {layer_idx}, Task: {task})")
    plt.xlabel("Synapse (Expert) Index")
    plt.ylabel("Token Index in Sequence")
    plt.show()

# Visualize the state before training
print("=== Before Training ===")
visualize_routing(model, title_prefix="[Before Training]")

## 4. रन ट्रेनिंग (कॉपी टास्क)
केवल कुछ युगों के लिए कॉपी कार्य (इनपुट को यथावत आउटपुट) पर प्रशिक्षित करें।

In [ ]:
print("Training started...")
model.train()

epochs = 300
batch_size = 32

for epoch in range(epochs):
    x, y = make_batch("copy", batch_size, 5, 10, device)
    
    optimizer.zero_grad()
    y_in = torch.cat([torch.full((y.size(0), 1), 1, dtype=torch.long, device=device), y[:, :-1]], dim=1)
    outputs, routing_weights, _ = model(x, y_in)
    
    # CrossEntropyLoss (reshape outputs to match)
    # outputs: [batch, seq, vocab], y: [batch, seq]
    loss = F.cross_entropy(outputs.reshape(-1, config.vocab_size), y.reshape(-1))
    
    # Add the router's load-balancing loss (so that specific experts aren't always used)
    lb_loss = load_balance_loss(routing_weights) * 0.1
    total_loss = loss + lb_loss
    
    total_loss.backward()
    optimizer.step()
    
    if (epoch + 1) % 50 == 0:
        print(f"Epoch {epoch+1}/{epochs} | Loss: {loss.item():.4f} | LB Loss: {lb_loss.item():.4f}")

print("Training finished!")

## 5. प्रशिक्षण के बाद रूटिंग और अनुमान की जाँच करें
तुलना करें कि प्रशिक्षण से पहले और बाद में रूटिंग (हीटमैप पूर्वाग्रह) कैसे बदल गया है।
जैसे-जैसे प्रशिक्षण आगे बढ़ता है, राउटर को जानबूझकर विशिष्ट विशेषज्ञों (सिनैप्स) को चुनना शुरू कर देना चाहिए, इसलिए आपको वजन को केंद्रित (विशेषीकृत) होते देखना चाहिए।

In [ ]:
print("=== After Training ===")
visualize_routing(model, title_prefix="[After Training]")

# Run inference to confirm the task is actually solved
model.eval()
x, y = make_batch("copy", batch_size=1, min_len=8, max_len=8, device=device)
with torch.no_grad():
    outputs, _, _ = model(x, torch.zeros_like(y))
    preds = outputs.argmax(dim=-1)
    
print("\n--- Inference Result ---")
print(f"Input (Target) : {y[0].tolist()}")
print(f"Prediction     : {preds[0].tolist()}")
if y[0].tolist() == preds[0].tolist():
    print("\n=> Success! The model has learned to copy.")
else:
    print("\n=> Failed... You may need a few more training epochs.")